# Emulador de stream de datos

In [ ]:
!python --version

In [1]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder\
    .master("spark://spark-master:7077")\
    .appName("Stream")\
    .config("spark.jars", "/opt/spark/jars/mysql-connector-j-8.4.0.jar")\
    .getOrCreate()  

In [4]:
import pyspark
import os

print("PySpark:", pyspark.__version__)
print("Spark:", spark.version)

PySpark: 3.5.1
Spark: 3.5.1


In [7]:
ultimo_id = 68884
consulta = f"""(
SELECT *
FROM orders
WHERE order_id > {ultimo_id}
) t
"""
df_orders = (
    spark.read
         .format("jdbc")
         .option("url", "jdbc:mysql://mysql:3306/retail_db")
         .option("dbtable", consulta)
         .option("user", "root")
         .option("password", "root")
         .option("driver", "com.mysql.cj.jdbc.Driver")
         .load()
)

df_orders.show()

[Stage 2:>                                                          (0 + 1) / 1]

+--------+-------------------+-----------------+------------+
|order_id|         order_date|order_customer_id|order_status|
+--------+-------------------+-----------------+------------+
|   68885|2026-08-04 18:18:59|             9303|     PENDING|
|   68886|2026-08-04 18:19:02|              977|     PENDING|
|   68887|2026-08-04 18:20:23|             1931|    COMPLETE|
|   68888|2026-08-04 18:20:28|             9271|  PROCESSING|
|   68889|2026-08-04 18:20:36|             3404|      CLOSED|
|   68890|2026-08-04 18:31:38|             2076|     PENDING|
|   68891|2026-08-04 18:31:43|             1488|      CLOSED|
+--------+-------------------+-----------------+------------+



# Kafka 

## A Creacion de un topico 

#### 1. ENtrar al contenedor de kafka 
docker exec -it kafka bash

#### 2. Enlistar topicos 
kafka-topics --bootstrap-server localhost:9092 --list

#### 3. Crea topico ventas 

kafka-topics --create --topic ventas --bootstrap-server localhost:9092 --partitions 2 --replication-factor 1

#### 4. Visualizdor en kafka como llegan los datos 

kafka-console-consumer --bootstrap-server kafka:9092 --topic ventas --from-beginning


## B en el notebook

In [8]:
!pip install kafka-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.1/614.1 kB 11.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
from kafka import KafkaProducer
import json
import time
import random
from datetime import datetime
#Crea conexión con Kafka
#Se conecta al broker Kafka:

producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda x: json.dumps(x).encode("utf-8")
)

while True:
    # Genera una venta ficticia cada 3 segundos
    venta = {
        "order_id": random.randint(10000,99999),
        "customer_id": random.randint(1,1000),
        "monto": round(random.uniform(10,500),2),
        "fecha": str(datetime.now())
    }
    
    # Envía el mensaje a Kafka
    producer.send(
        "ventas",
        value=venta
    )


    print("Enviado:", venta)


    time.sleep(8)

Enviado: {'order_id': 41608, 'customer_id': 915, 'monto': 259.16, 'fecha': '2026-08-04 18:51:18.417140'}
Enviado: {'order_id': 39365, 'customer_id': 709, 'monto': 228.73, 'fecha': '2026-08-04 18:51:26.527659'}
Enviado: {'order_id': 85560, 'customer_id': 883, 'monto': 137.62, 'fecha': '2026-08-04 18:51:34.536054'}
Enviado: {'order_id': 51361, 'customer_id': 885, 'monto': 211.43, 'fecha': '2026-08-04 18:51:42.544834'}
Enviado: {'order_id': 22720, 'customer_id': 339, 'monto': 427.71, 'fecha': '2026-08-04 18:51:50.553678'}
Enviado: {'order_id': 23704, 'customer_id': 435, 'monto': 159.53, 'fecha': '2026-08-04 18:51:58.561900'}
Enviado: {'order_id': 21717, 'customer_id': 663, 'monto': 251.04, 'fecha': '2026-08-04 18:52:06.569931'}
Enviado: {'order_id': 89485, 'customer_id': 387, 'monto': 174.28, 'fecha': '2026-08-04 18:52:14.578047'}
Enviado: {'order_id': 67896, 'customer_id': 24, 'monto': 339.92, 'fecha': '2026-08-04 18:52:22.588463'}
Enviado: {'order_id': 24974, 'customer_id': 814, 'monto'

KeyboardInterrupt: 

## Kafka guarda el evento en el tópico: ordenes_ventas

In [11]:
!pip install kafka-python mysql-connector-python


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


#### 1. ENtrar al contenedor de kafka 
    docker exec -it kafka bash

#### 2. Enlistar topicos 
    kafka-topics --bootstrap-server localhost:9092 --list

#### 3. Crea topico order , orders_items

    kafka-topics --create --topic orders_topic --bootstrap-server kafka:9092 --partitions 2 --replication-factor 1
    
    kafka-topics --create --topic order_items_topic --bootstrap-server kafka:9092 --partitions 2 --replication-factor 1
    
#### 4. Crear  mysql_kafka_producer.py

    

#### 5. Visualizdor en kafka como llegan los datos 

    kafka-console-consumer --bootstrap-server kafka:9092 --topic ordenes_ventas --from-beginning